# FinReason — DPO Training (Colab Pro A100)
Generate preference pairs from SFT checkpoint, then run DPO alignment.

**Prerequisite:** Run `colab_sft.ipynb` first. SFT checkpoint must exist in Drive.

**Runtime:** A100 GPU required.
Runtime → Change runtime type → A100

In [ ]:
# Verify GPU
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q torch transformers datasets peft trl bitsandbytes accelerate wandb huggingface_hub PyYAML scipy

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/finreason/checkpoints/dpo', exist_ok=True)

In [ ]:
# Clone repo
!git clone https://github.com/YOUR_USERNAME/finreason.git
%cd finreason

In [ ]:
# Restore data from Drive (already prepared in colab_sft)
!cp -r /content/drive/MyDrive/finreason/data/ data/

In [ ]:
# Login
from huggingface_hub import login
import wandb
login()
wandb.login()

In [ ]:
# Point configs to Drive paths
import yaml

with open('configs/sft.yaml') as f:
    sft_cfg = yaml.safe_load(f)
sft_cfg['training']['output_dir'] = '/content/drive/MyDrive/finreason/checkpoints/sft'
with open('configs/sft.yaml', 'w') as f:
    yaml.dump(sft_cfg, f)

with open('configs/dpo.yaml') as f:
    dpo_cfg = yaml.safe_load(f)
dpo_cfg['training']['sft_checkpoint'] = '/content/drive/MyDrive/finreason/checkpoints/sft'
dpo_cfg['training']['output_dir'] = '/content/drive/MyDrive/finreason/checkpoints/dpo'
with open('configs/dpo.yaml', 'w') as f:
    yaml.dump(dpo_cfg, f)

In [ ]:
# Generate DPO preference pairs from SFT model (~1 hour)
# Runs SFT model 5x per question, finds correct vs wrong pairs
!python scripts/prepare_data.py --dpo --n_runs 5

In [ ]:
# Check how many pairs were generated
!wc -l data/dpo_pairs.jsonl

# Save pairs to Drive
!cp data/dpo_pairs.jsonl /content/drive/MyDrive/finreason/data/

In [ ]:
# Run DPO training (~1 hour on A100)
!python scripts/train_dpo.py --config configs/dpo.yaml

In [ ]:
# Verify DPO checkpoint saved
!ls /content/drive/MyDrive/finreason/checkpoints/dpo/

In [ ]:
# Point eval config to Drive checkpoints
with open('configs/eval.yaml') as f:
    eval_cfg = yaml.safe_load(f)
eval_cfg['model']['sft_checkpoint'] = '/content/drive/MyDrive/finreason/checkpoints/sft'
eval_cfg['model']['dpo_checkpoint'] = '/content/drive/MyDrive/finreason/checkpoints/dpo'
with open('configs/eval.yaml', 'w') as f:
    yaml.dump(eval_cfg, f)

In [ ]:
# Run 3-stage evaluation
!python scripts/evaluate.py --config configs/eval.yaml

In [ ]:
# Copy results to Drive
!cp -r results/ /content/drive/MyDrive/finreason/results/

In [ ]:
# Push to HuggingFace Hub
# Replace YOUR_USERNAME with your HuggingFace username
!python scripts/push_to_hub.py --repo YOUR_USERNAME/finreason-qwen2.5-7b-dpo

## Done!
- DPO checkpoint saved to Google Drive
- Eval results saved to `results/`
- Model live on HuggingFace Hub

Next: commit results JSON to GitHub, write README with real numbers.